In [0]:
from pyspark.sql import SparkSession 
spark = SparkSession.builder.appName("Project Dataframe").getOrCreate()


In [0]:
path=r'abfss://healthcareproject@storageaccgroupc.dfs.core.windows.net/bronze/dbo.diagnoses.csv'
diagnoses_df =spark.read.csv(path,header=True,inferSchema=True)
diagnoses_df.count()
diagnoses_df.printSchema()


In [0]:
#Remove duplicates
diagnoses_dropduplicates =diagnoses_df.dropDuplicates(['diagnosis_id'])
diagnoses_dropduplicates.count()#300 was duplicate

In [0]:
#Standardization for icd10_code colum
from pyspark.sql.functions import upper, trim, col, regexp_replace

diagnoses_icd10 = (
    diagnoses_dropduplicates
    .withColumn(
        "icd10_code_std",
        regexp_replace(
            upper(trim(col("icd10_code"))),
            r'^([A-Z][0-9]{2})([0-9]+)$',
            r'\1.\2'
        )
    )
    .drop("icd10_code")
)


In [0]:
#ICD10 description cleaning
from pyspark.sql.functions import initcap
diagnoses_icd10desc = diagnoses_icd10 \
    .withColumn("icd10_desc_clean",
                initcap(trim(col("icd10_description")))) \
    .drop("icd10_description")

diagnoses_icd10desc.display()

In [0]:
#SNOMED cleaning
diagnoses_snomed =( diagnoses_icd10desc.withColumn(
        "snomed_code_clean",
        regexp_replace(col("snomed_code"), "[^0-9]", "").cast("long")
    )
    .drop("snomed_code")
    )
diagnoses_snomed.display()

In [0]:
#Onset date,resolution_date standardization and using date columns created derived column as diagnosis_duration_days
from pyspark.sql.functions import try_to_date,coalesce,datediff
diagnoses_Date =(diagnoses_snomed.withColumn(
        "onset_date_std",
        coalesce(
            try_to_date(col("onset_date"), "dd-MM-yyyy"),
            try_to_date(col("onset_date"), "yyyy-MM-dd"),
            try_to_date(col("onset_date"), "MM-dd-yyyy"),
            try_to_date(col("onset_date"), "dd/MM/yyyy")
          )
        )
        .withColumn("resolution_date_std",
        coalesce(
            try_to_date(col("resolution_date"), "dd-MM-yyyy"),
            try_to_date(col("resolution_date"), "yyyy-MM-dd"),
            try_to_date(col("resolution_date"), "MM-dd-yyyy"),
            try_to_date(col("resolution_date"), "dd/MM/yyyy")
          )
        )
        .drop("onset_date")
        .drop("resolution_date")
        .withColumn("diagnosis_duration_days",
            datediff(col("resolution_date_std"), col("onset_date_std"))
)
)


diagnoses_Date.display()


In [0]:
#Diagnosis type standardization
from pyspark.sql.functions import when
diagnoses_std =( diagnoses_Date.withColumn(
        "diagnosis_type_std",
        when(col("diagnosis_type").isNull(), "Unknown")
        .otherwise(initcap(trim(col("diagnosis_type"))))
    )
    .withColumn(
        "diagnosis_status_std",
        when(col("diagnosis_status").isNull(), "Unknown")
        .otherwise(initcap(trim(col("diagnosis_status"))))
    )
    .withColumn(
        "severity_std",
        when(col("severity").isNull(), "Unknown")
        .otherwise(initcap(trim(col("severity"))))
    )
    .drop("diagnosis_type")\
    .drop("diagnosis_status")\
    .drop("severity")
)
diagnoses_std.display()


In [0]:
#Severity numeric mapping -created severity_score column from severity_std column
from pyspark.sql.functions import when
diagnoses_severity=diagnoses_std.withColumn(
        "severity_score",
        when(col("severity_std") == "Mild", 1)
        .when(col("severity_std") == "Moderate", 2)
        .when(col("severity_std") == "Severe", 3)
        .when(col("severity_std") == "Critical", 4)
    )


#diagnoses_severity.display()
diagnoses_severity.select("severity_score").distinct().show()

In [0]:
#Chronic flag normalization ,Boolean Normalisation
diagnoses_chronic_flag=diagnoses_severity.withColumn(
        "chronic_flag_std",
        when(col("chronic_flag").isin("Yes", "Y", "1"), 1)
        .when(col("chronic_flag").isin("No", "N", "0"), 0)
        .otherwise(0)
    )

In [0]:
#Diagnosis confidence standardization
from pyspark.sql.functions import lower, trim, col, when
diagnoses_stand=(diagnoses_chronic_flag.withColumn(
        "confidence_std",
        when(col("diagnosis_confidence").isNull(), "unknown")
        .otherwise(lower(trim(col("diagnosis_confidence"))))
    )
    .drop("diagnosis_confidence")            
)


diagnoses_stand.display()

In [0]:
 #Derived – ICD Chapter Column
from pyspark.sql.functions import substring
diagnoses_Chapter=diagnoses_stand.withColumn(
        "icd10_chapter",
        substring(col("icd10_code_std"), 1, 1)#(position,length)
    )
diagnoses_Chapter.display()

In [0]:
#Derived flag for mental health
diagnoses_derived_flag = diagnoses_Chapter.withColumn(
        "is_mental_health_dx",
        when(col("icd10_code_std").startswith("F"), 1)
        .otherwise(0)
    )


In [0]:
#Provider cleanup
diagnoses_provider = diagnoses_derived_flag.withColumn(
        "diagnosed_by_provider",
        trim(col("diagnosed_by_provider"))
    )  

diagnoses_provider.count()


In [0]:
#Ingestion date extraction
from pyspark.sql.functions import try_to_date
diagnoses_ingestion_date = diagnoses_provider.withColumn(
        "ingestion_date",
        try_to_date(col("ingestion_timestamp"))
    )

diagnoses_ingestion_date.display()



In [0]:
#Data-quality flag
df_diagnoses = diagnoses_ingestion_date.withColumn(
        "dq_dx_flag",
        when(
            col("icd10_code_std").isNull()
            | col("onset_date_std").isNull()
            | (col("diagnosis_type_std") == "Unknown"),
            1
        ).otherwise(0)
    )  

df_diagnoses.display()

In [0]:
# ----------------------------------------
#  Write to Silver layer in delta format
# ----------------------------------------


TABLE_NAME = "diagnoses"  

silver_table_path = f"abfss://healthcareproject@storageaccgroupc.dfs.core.windows.net/silver/{TABLE_NAME}"

# ── OVERWRITE (Full Load) ────────────────────────────────────
df_diagnoses.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_table_path)

print(f"✅ Data written to Silver layer: {silver_table_path}")